**Model:** Linear Discriminant Analysis 

**Score:**   
LDA: 0.688.  
Best Parameters: {'shrinkage': 0, 'solver': 'lsqr'}.    

LDA with PCA: 0.688.  
Best Parameters: {'lda__shrinkage': 'auto', 'lda__solver': 'lsqr', 'pca__n_components': 20}.  

LDA without elo features: 0.651.  
Best Parameters: {'shrinkage': 0.01, 'solver': 'lsqr'}.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Preparation

In [2]:
data = pd.read_csv('match_data_300_tourns_modified.csv')

In [3]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle=False)


In [4]:
#Create predictors and targets for training and cross-validation set
y_train = data_train['match_result']
y_test = data_test['match_result']

#To get the predictor, we exclude players' names (Strings), date, tournament ids and match results.
X_train = data_train.drop(['match_result', 'win_percentage', 'score1', 'score2', 'player1', 'player2', 'tournament_id', 'date'],
                           axis = 1)
X_test = data_test.drop(['match_result', 'win_percentage', 'score1', 'score2', 'player1', 'player2', 'tournament_id', 'date'],
                           axis = 1)

***

## Performance Metrics
Because the two classes in the target are symmetric (swapping player1 and player2 will exchange positive and negative but still represents the same match), we won't consider metrics such as presision, specificity and sensitivity since they are the same as accuracy score. We will only consider accuracy score.

In [5]:
#Import metrics
from sklearn.metrics import accuracy_score

In [6]:
#Create dictionaries to store the metrics.
accuracy_scores = {}

#Create a list to store all the models we consider.
models = {}

In [7]:
def print_avg_cv_metrics(model, model_name):
    """
    Given a model, computes and stores accuracy scores on the test set.
    """

    #Create an empty array to store the scores.
    print('Currently working on ' + model_name + '.')

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    score = accuracy_score(y_test, y_pred)

    print('The accuracy score of ' + model_name + ' is:', score)
    
    #Record the scores
    accuracy_scores[model_name] = score

    models[model_name] = model


***

### Linear Discriminant Analysis

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV

In [9]:
lda = LinearDiscriminantAnalysis()

param_grid = {
    "solver": ['lsqr', 'eigen'],
    "shrinkage": ['auto', 0, 0.01, 0.1, 1]
}

grid_search1 = GridSearchCV(lda,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)

grid_search1.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LinearDiscriminantAnalysis(),
             param_grid={'shrinkage': ['auto', 0, 0.01, 0.1, 1],
                         'solver': ['lsqr', 'eigen']},
             scoring='accuracy')

In [10]:
# Print the parameters with best performance in the cross_validations and the corresponding score.
print(grid_search1.best_params_)
print(grid_search1.best_score_)

{'shrinkage': 0, 'solver': 'lsqr'}
0.6917725729399917


In [11]:
# Use the model to make prediction on the test set and print the score.
model1 = grid_search1.best_estimator_
print_avg_cv_metrics(model1, 'LDA')

Currently working on LDA.
The accuracy score of LDA is: 0.6876176683562636


***

### Linear Discriminant Analysis with PCA

In [12]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [13]:
lda = LinearDiscriminantAnalysis()
pca = PCA()
scaler = StandardScaler()

lda2 = Pipeline(
    [('scale', scaler),
     ('pca', pca),
     ('lda', lda)]
)

param_grid = {
    "lda__solver": ['lsqr', 'eigen'],
    "lda__shrinkage": ['auto', 0, 0.01, 0.1, 1],
    "pca__n_components": [5, 10, 15, 20]
}

grid_search3 = GridSearchCV(lda2,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)

grid_search3.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('pca', PCA()),
                                       ('lda', LinearDiscriminantAnalysis())]),
             param_grid={'lda__shrinkage': ['auto', 0, 0.01, 0.1, 1],
                         'lda__solver': ['lsqr', 'eigen'],
                         'pca__n_components': [5, 10, 15, 20]},
             scoring='accuracy')

In [14]:
# Print the parameters with best performance in the cross_validations and the corresponding score.
print(grid_search3.best_params_)
print(grid_search3.best_score_)

{'lda__shrinkage': 'auto', 'lda__solver': 'lsqr', 'pca__n_components': 20}
0.6913018142943281


In [15]:
# Use the model to make prediction on the test set and print the score.
model3 = grid_search1.best_estimator_
print_avg_cv_metrics(model3, 'LDA with PCA')

Currently working on LDA with PCA.
The accuracy score of LDA with PCA is: 0.6876176683562636


***

### Linear discriminant analysis without elo related features

In [16]:
#Create predictors and targets for training and cross-validation set
y_train = data_train['match_result']
y_test = data_test['match_result']

#To get the predictor, we exclude players' names (Strings), date, tournament ids and match results, and elo features.
X_train = data_train.drop(['match_result', 'win_percentage', 'score1', 'score2',
                            'player1', 'player2', 'tournament_id', 'date',
                            'player1_elo', 'player2_elo', 'elo_match_win_rate', 'elo_frame_win_rate'],
                           axis = 1)
X_test = data_test.drop(['match_result', 'win_percentage', 'score1', 'score2',
                            'player1', 'player2', 'tournament_id', 'date',
                            'player1_elo', 'player2_elo', 'elo_match_win_rate', 'elo_frame_win_rate'],
                           axis = 1)



In [17]:
lda = LinearDiscriminantAnalysis()

param_grid = {
    "solver": ['lsqr', 'eigen'],
    "shrinkage": ['auto', 0, 0.01, 0.1, 1]
}

grid_search2 = GridSearchCV(lda,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)

grid_search2.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LinearDiscriminantAnalysis(),
             param_grid={'shrinkage': ['auto', 0, 0.01, 0.1, 1],
                         'solver': ['lsqr', 'eigen']},
             scoring='accuracy')

In [18]:
# Print the parameters with best performance in the cross_validations and the corresponding score.
print(grid_search2.best_params_)
print(grid_search2.best_score_)

{'shrinkage': 0.01, 'solver': 'lsqr'}
0.6769980660166037


In [19]:
# Use the model to make prediction on the test set and print the score.
model2 = grid_search2.best_estimator_
print_avg_cv_metrics(model2, 'LDA without elo features')

Currently working on LDA without elo features.
The accuracy score of LDA without elo features is: 0.6512671976828385


***

In [20]:
print(accuracy_scores)

{'LDA': 0.6876176683562636, 'LDA with PCA': 0.6876176683562636, 'LDA without elo features': 0.6512671976828385}
